In [1]:
import os
import caldav
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
username = os.getenv("ICLOUD_USERNAME")
password = os.getenv("ICLOUD_APP_PASSWORD")

print("Username loaded:", bool(username))
print("Password loaded:", bool(password))

Username loaded: True
Password loaded: True


In [3]:
client = caldav.DAVClient(
    url="https://caldav.icloud.com",
    username=username,
    password=password,
)

principal = client.principal()
print(principal)

Max A. H. Hartvigsen


In [5]:
calendars = principal.calendars()

for calendar in calendars:
    print(calendar.get_display_name())

Family ⚠️
Family Calendar
Isabella-Timetable Perse 
Sjii and Perse school cal
Reminders ⚠️
Private bork 
Planning


In [6]:
from datetime import datetime
from zoneinfo import ZoneInfo

start = datetime(
    2026, 10, 8, 19, 0,
    tzinfo=ZoneInfo("Asia/Singapore")
)

end = datetime(
    2026, 10, 8, 20, 30,
    tzinfo=ZoneInfo("Asia/Singapore")
)

wanted = {"Family Calendar", "Private bork"}

selected_calendars = [
    calendar
    for calendar in calendars
    if calendar.get_display_name().strip() in wanted
]

print([c.get_display_name() for c in selected_calendars])

['Family Calendar', 'Private bork ']


In [7]:
for calendar in selected_calendars:
    print("\nCALENDAR:", calendar.get_display_name())

    events = calendar.search(
        start=start,
        end=end,
        event=True,
        expand=True,
    )

    print("Events found:", len(events))

    for event in events:
        vevent = event.icalendar_component
        print(
            "-",
            vevent.get("summary"),
            vevent.get("dtstart"),
            vevent.get("dtend"),
        )


CALENDAR: Family Calendar
Events found: 3
- Mama in HK vDDDTypes(2026-10-05, Parameters({'VALUE': 'DATE'})) vDDDTypes(2026-10-09, Parameters({'VALUE': 'DATE'}))
- Mama BKK vDDDTypes(2026-10-08, Parameters({'VALUE': 'DATE'})) vDDDTypes(2026-10-12, Parameters({'VALUE': 'DATE'}))
- Flight: SQ 714 from SIN to BKK vDDDTypes(2026-10-08 17:30:00+08:00, Parameters({'TZID': 'Asia/Singapore'})) vDDDTypes(2026-10-08 19:00:00+07:00, Parameters({'TZID': 'Asia/Bangkok'}))

CALENDAR: Private bork 
Events found: 0


In [8]:
from datetime import date, datetime
from zoneinfo import ZoneInfo

query_timezone = ZoneInfo("Asia/Singapore")

conflicts = []

for calendar in selected_calendars:
    for event in calendar.search(
        start=start,
        end=end,
        event=True,
        expand=True,
    ):
        vevent = event.icalendar_component

        summary = str(vevent.get("summary", "Untitled event"))

        dtstart = vevent.decoded("dtstart")
        dtend = vevent.decoded("dtend")

        # Ignore all-day events for availability checking
        if isinstance(dtstart, date) and not isinstance(dtstart, datetime):
            continue

        # Convert event times into the timezone of the availability query
        event_start = dtstart.astimezone(query_timezone)
        event_end = dtend.astimezone(query_timezone)

        # Confirm there is a real overlap
        if event_start < end and event_end > start:
            conflicts.append({
                "title": summary,
                "calendar": calendar.get_display_name().strip(),
                "start": event_start.isoformat(),
                "end": event_end.isoformat(),
            })

conflicts

[{'title': 'Flight: SQ 714 from SIN to BKK',
  'calendar': 'Family Calendar',
  'start': '2026-10-08T17:30:00+08:00',
  'end': '2026-10-08T20:00:00+08:00'}]

In [9]:
availability = {
    "available": len(conflicts) == 0,
    "start": start.isoformat(),
    "end": end.isoformat(),
    "timezone": "Asia/Singapore",
    "conflicts": conflicts,
}

availability

{'available': False,
 'start': '2026-10-08T19:00:00+08:00',
 'end': '2026-10-08T20:30:00+08:00',
 'timezone': 'Asia/Singapore',
 'conflicts': [{'title': 'Flight: SQ 714 from SIN to BKK',
   'calendar': 'Family Calendar',
   'start': '2026-10-08T17:30:00+08:00',
   'end': '2026-10-08T20:00:00+08:00'}]}